# 14 — Pipeline discovery and the MCP connector

**The concept:** everything in [10 — Analysis](10-analysis.ipynb) works on a `Pipeline` *object*. To
use it on a **file** — which is what a coding agent has — something must find the pipeline inside
that file without running the file's demo code.

That is `load_pipeline`, and the safety rule it follows matters more than the feature.

In [1]:
import pathlib, tempfile
from smartmdao.mcp.loader import load_pipeline, PipelineLoadError

workspace = pathlib.Path(tempfile.mkdtemp())
shapes = workspace / "shapes.py"
shapes.write_text('''# Five shapes a model file can take, and what the loader makes of each.
from smartmdao import Pipeline

# 1. A module-level instance: read directly, nothing called.
module_level = Pipeline()


@module_level.step(outputs=["b"])
def double(a: float) -> float:
    return a * 2


# 2. A factory declaring -> Pipeline: called, because it says what it returns.
def build_pipeline() -> Pipeline:
    pipeline = Pipeline()

    @pipeline.step(outputs=["d"])
    def triple(c: float) -> float:
        return c * 3

    return pipeline


# 3. A factory that needs arguments: reported WITH the argument names.
def build_parametrised(scale: float) -> Pipeline:
    pipeline = Pipeline()

    @pipeline.step(outputs=["f"])
    def scaled(e: float) -> float:
        return e * scale

    return pipeline


# 4. An unannotated factory: NOT called speculatively.
def make_something():
    return Pipeline()


# 5. A demo that builds a pipeline and never returns it: unreachable by design.
def run_demo():
    pipeline = Pipeline()

    @pipeline.step(outputs=["h"])
    def quadruple(g: float) -> float:
        return g * 4

    return pipeline.run(g=1.0)
''')

print(shapes.read_text())

# Five shapes a model file can take, and what the loader makes of each.
from smartmdao import Pipeline

# 1. A module-level instance: read directly, nothing called.
module_level = Pipeline()


@module_level.step(outputs=["b"])
def double(a: float) -> float:
    return a * 2


# 2. A factory declaring -> Pipeline: called, because it says what it returns.
def build_pipeline() -> Pipeline:
    pipeline = Pipeline()

    @pipeline.step(outputs=["d"])
    def triple(c: float) -> float:
        return c * 3

    return pipeline


# 3. A factory that needs arguments: reported WITH the argument names.
def build_parametrised(scale: float) -> Pipeline:
    pipeline = Pipeline()

    @pipeline.step(outputs=["f"])
    def scaled(e: float) -> float:
        return e * scale

    return pipeline


# 4. An unannotated factory: NOT called speculatively.
def make_something():
    return Pipeline()


# 5. A demo that builds a pipeline and never returns it: unreachable by design.
def run_demo():
    pipe

## The safety rule

**A function is only ever called when it *declares* `-> Pipeline`, or when the caller names it.**
Nothing is called speculatively to see what it returns — a module's `run_demo()` would execute the
entire study, and *no discipline is ever invoked* is the promise the whole analysis layer rests on.

Calling a factory registers steps. It does not evaluate them.

In [2]:
loaded = load_pipeline(str(shapes), variable="build_pipeline")
print("source:", loaded.source)
print("steps: ", [s.name for s in loaded.pipeline.steps])

source: factory
steps:  ['triple']


`LoadedPipeline.source` records whether the pipeline was **read** or **called** — because how a
result was obtained is part of what the engineer needs to know.

In [3]:
read_directly = load_pipeline(str(shapes), variable="module_level")
print("module_level  ->", read_directly.source)
print("build_pipeline ->", loaded.source)

module_level  -> variable
build_pipeline -> factory


## Ambiguity is reported, not guessed

A file with several candidates does not get a silent choice made for it. Every candidate is listed,
instances and factories alike.

In [4]:
try:
    load_pipeline(str(shapes))
except PipelineLoadError as error:
    print(f"PipelineLoadError: {error}")

PipelineLoadError: shapes.py offers several pipelines. Pass `variable` to choose one. Pipelines: ['module_level']. Factories: ['build_parametrised() [needs: scale]', 'build_pipeline()'].


## A factory needing arguments is reported *with the argument names*

Generic failures make you go and read the file. This one tells you what to supply.

In [5]:
try:
    load_pipeline(str(shapes), variable="build_parametrised")
except PipelineLoadError as error:
    print(f"PipelineLoadError: {error}")

PipelineLoadError: 'build_parametrised' in shapes.py needs argument(s) ['scale'], so it cannot be called automatically. Give them defaults, or wrap it in a zero-argument factory.


## What is unreachable, and why that is correct

A pipeline built inside a function body and **never returned** cannot be found. There is nothing to
call and nothing to read; reaching it would mean running the function, which is the one thing this
layer promises not to do.

That shape is normal for a *script* and unusual for a *model*. Measured across this repository's own
scripts when the loader was built: 9 auto-discovered, 1 more by name, 14 unreachable.

In [6]:
try:
    load_pipeline(str(shapes), variable="run_demo")
except PipelineLoadError as error:
    print(f"PipelineLoadError: {error}")
except Exception as error:
    print(f"{type(error).__name__}: {error}")

PipelineLoadError: run_demo() in shapes.py is annotated as returning a Pipeline but returned dict.


**To make your model readable:** expose it as a module-level instance, or as a factory
annotated `-> Pipeline` that is callable with no arguments.

## The MCP tools

The connector is a **verifier and an oracle, not an author**. The client driving it is already a
capable language model; what it cannot do is compute the structure of the code it just wrote.

Each tool takes a *path* and wraps the analysis layer.

In [7]:
from smartmdao.mcp.handlers import (
    analyze_pipeline, validate_pipeline, explain_pipeline,
)

model = workspace / "sellar.py"
model.write_text(
    "from smartmdao import Pipeline, HybridSolver\n\n"
    "def build_pipeline() -> Pipeline:\n"
    "    p = Pipeline(solver=HybridSolver())\n\n"
    "    @p.step(outputs=['y1'])\n"
    "    def discipline_1(z: float, y2: float) -> float:\n"
    "        return z**2 - 0.2 * y2\n\n"
    "    @p.step(outputs=['y2'])\n"
    "    def discipline_2(y1: float) -> float:\n"
    "        return abs(y1) ** 0.5\n\n"
    "    return p\n"
)

analysis = analyze_pipeline(str(model), inputs=["z"])
for key in ("execution_order", "cycles", "initial_guesses_required", "recommended_solver"):
    if key in analysis:
        print(f"{key}: {analysis[key]}")

execution_order: ['discipline_1', 'discipline_2']
cycles: [{'steps': ['discipline_1', 'discipline_2'], 'feedback_variables': ['y1', 'y2']}]
initial_guesses_required: [{'variable': 'y2', 'consumed_by': 'discipline_1'}]
recommended_solver: HybridSolver


In [8]:
print("validate:", validate_pipeline(str(model), inputs=["z"]))

validate: {'ok': True, 'pipeline': 'build_pipeline', 'source': 'factory', 'path': '/tmp/tmp5pyplj8f/sellar.py', 'inputs_used': {'requested': ['z'], 'found_in_source': []}, 'valid': False, 'counts': {'error': 1}, 'findings': [{'code': 'initial-guess-required', 'severity': 'error', 'message': "'discipline_1' consumes 'y2' before anything produces it, so the first sweep has nothing to read. Ordering here is decided by HybridSolver, so renaming a step or swapping solvers can change which variable this is. Pass y2=... to run().", 'step': 'discipline_1', 'variable': 'y2'}]}


In [9]:
print(explain_pipeline(str(model), inputs=["z"])["explanation"])

Pipeline with 2 step(s): discipline_1, discipline_2.

Recommended solver: HybridSolver
  1 feedback loop(s) detected; DAGSolver would raise. HybridSolver runs acyclic steps once and iterates only the cyclic blocks.

External inputs: z

Feedback loops (1):
  1. discipline_1 -> discipline_2
     coupling on: y1, y2

Needs initial values for: y2 (for discipline_1)

Execution order: discipline_1 -> discipline_2

Findings (1):
  ERROR: 'discipline_1' consumes 'y2' before anything produces it, so the first sweep has nothing to read. Ordering here is decided by HybridSolver, so renaming a step or swapping solvers can change which variable this is. Pass y2=... to run(). [discipline_1]


## The verification loop

This is the workflow the connector exists to support, and the order matters:

```
agent drafts code
  -> analyze_pipeline    order, cycles, feedback vars, recommended solver
  -> validate_pipeline   every structural problem at once
  -> agent fixes
  -> run_pipeline        smoke rung: proves it executes, measures the unit cost
  -> compare_runs        if this was a translation
  -> render_pipeline_diagram
```

The first two are **free** — no discipline is called — which is why they come first. Generation was
never the bottleneck; **verification** is.

---

**Next:** [15 — Pitfalls](15-pitfalls.ipynb).